# Week 4 Lab — Raster values, georeferencing and real data

**Geo-Data and Geo-Data Management · 2026W**

**Main question:** How can we use the same core Rasterio workflow with a synthetic grid and two different kinds of real raster?

The lab has three parts:

- **Part A — Guided synthetic digital elevation model (DEM) (sections 1–6).** Create a 4 × 4 fictional elevation grid and practise indexing, georeferencing, GeoTIFF writing/reading, NoData and a georeferenced plot.
- **Part B — Guided real Vienna DEM (sections 7–9).** Use an authentic elevation GeoTIFF supplied by the instructor. Inspect metadata, map real terrain and calculate valid elevation statistics.
- **Part C — Independent checkpoint: ESA WorldCover (sections 10–11).** Use a **second authentic GeoTIFF** containing land-cover categories. Independently inspect it, summarise category counts/shares and make a categorical map.

Part A uses a synthetic DEM. The analyses in Parts B and C use *different real datasets*. Section 10 also includes a labelled synthetic 3 × 3 land-cover toy to practise counts and the plotting template before the independent real-data analysis. Each source file has its own CRS, resolution, extent, NoData rules and value meaning. Do not copy the synthetic DEM metadata into either real raster, and do not assume the two real rasters are aligned.

**By the end:** explain georeferenced raster values, inspect an unfamiliar GeoTIFF, distinguish quantitative from categorical data, handle invalid cells and choose an appropriate summary and map.


## Before you start

Open the notebook in the course Jupyter environment and run the Guided cells in order. Packages: NumPy, Rasterio and Matplotlib. Generated files are saved in `week4_outputs/`. Figures are displayed in the notebook; keep their outputs when saving your `.ipynb`.

**Instructor-supplied authentic input files (required):**

Upload or place these **three separate files together in the same folder**. Do not create subfolders:

```text
W04_Vienna_Lab.ipynb
W04_Vienna_DEM_Clip.tif          # Part B · Guided real DEM
W04_Vienna_WorldCover_Clip.tif   # Part C · Independent checkpoint
```

The notebook never invents replacement observations if a file is missing. You can complete Part A without real files, but Parts B and C need their respective authentic files.

- **Vienna DEM:** [Stadt Wien Digitales Geländemodell](https://www.wien.gv.at/stadtplanung/digitales-gelaendemodell-daten-beziehen), attribution: *Datenquelle: Stadt Wien - data.wien.gv.at* (CC BY 4.0). Follow the delivered file's CRS and the source's elevation reference (the original product documentation describes GK-M34 / EPSG:31256 and Wiener Null).
- **WorldCover:** [ESA WorldCover 2021 v200](https://esa-worldcover.org/en/data-access), attribution: *© ESA WorldCover project 2021 / Contains modified Copernicus Sentinel data (2021) processed by ESA WorldCover consortium* (CC BY 4.0). The original Map layer is in EPSG:4326; the delivered clip must be inspected independently.

**Week 4 scope:** read and analyse the two real rasters separately. Raster alignment, resampling, overlay and area calculation belong to later lessons.


## Part A — Guided synthetic DEM

### 1 · Import the tools

In [ ]:
from pathlib import Path
import numpy as np
import rasterio
from rasterio.transform import from_origin
from rasterio.plot import show
import matplotlib.pyplot as plt

workdir = Path("week4_outputs")
workdir.mkdir(exist_ok=True)
print("NumPy:", np.__version__)
print("Rasterio:", rasterio.__version__)

## 2 · Read a small NumPy grid

The example is a **4 × 4 synthetic elevation raster**. The physical unit is **metres**. The value `-9999` is a **declared missing-data sentinel**, not a low elevation. All other cells are valid.

In [ ]:
arr = np.array([
    [210, 212, 218, -9999],
    [208, 215, 225, 229],
    [205, 213, 240, 235],
    [200, 208, 219, 230],
], dtype=np.float32)

nodata = -9999.0
print(arr)
print("shape:", arr.shape)
print("dtype:", arr.dtype)
print("arr[2, 1]:", arr[2, 1])
print("raw minimum:", arr.min(), "(includes NoData)")
print("raw maximum:", arr.max())

A NumPy index is written **`[row, column]`** and begins at zero. For example, `arr[2,1]` stores the synthetic elevation **213 m**. It has an array position but no geographic coordinate yet.

**Quick check:** What is the value of `arr[0,0]`? Where is `240` in the array?

## 3 · Attach the known georeferencing

This north-up teaching grid has:
- upper-left **corner**: easting `600000 m`, northing `5340000 m`;
- cell size: `100 m × 100 m`;
- CRS: **EPSG:32633** (WGS 84 / UTM zone 33N; projected coordinates in **metres**).

The transform maps pixel position to projected coordinates. The CRS tells us what those projected coordinates mean.

In [ ]:
crs = "EPSG:32633"
transform = from_origin(600000, 5340000, 100, 100)

# Hand calculation for the centre of cell (row=2, col=1)
row, col = 2, 1
x_manual = 600000 + (col + 0.5) * 100
y_manual = 5340000 - (row + 0.5) * 100
print("hand-calculated cell centre (m):", (x_manual, y_manual))

The `+0.5` offsets move from the cell corner to its **centre**. For this north-up grid, columns move east (higher easting) and rows move south (lower northing). A centre coordinate is not the same as the cell's whole footprint or its positional accuracy.

## 4 · Save and reopen a one-band GeoTIFF

Rasterio writes the numeric values together with their **width, height, data type, CRS, transform, band count and NoData setting**.

In [ ]:
tif_path = workdir / "elevation_demo.tif"
with rasterio.open(
    tif_path,
    "w",
    driver="GTiff",
    height=arr.shape[0],
    width=arr.shape[1],
    count=1,
    dtype=arr.dtype,
    crs=crs,
    transform=transform,
    nodata=nodata,
) as dst:
    dst.write(arr, 1)
print("Saved:", tif_path)

In [ ]:
with rasterio.open(tif_path) as src:
    data = src.read(1)  # Rasterio bands are 1-based
    print("read band shape:", data.shape)
    print("width, height, count:", src.width, src.height, src.count)
    print("CRS:", src.crs)
    print("resolution (m):", src.res)
    print("bounds (m):", src.bounds)
    print("NoData:", src.nodata)
    print("data type:", src.dtypes[0])
    print("profile:", src.profile)

Reading `src.read(1)` selects **band 1**, but indexing the returned `data[row, col]` starts from zero. `src.bounds` are reported as `(left, bottom, right, top)` in projected metres; `src.res` contains the pixel dimensions. Here the expected bounds are **(600000, 5339600, 600400, 5340000)**.

## 5 · Locate a cell and exclude NoData

We can verify the hand-calculated centre with the dataset's `.xy()` method. The minimum must be calculated from **valid** cells, not from the raw array.

In [ ]:
with rasterio.open(tif_path) as src:
    x_rasterio, y_rasterio = src.xy(2, 1)
print("Rasterio centre (m):", (x_rasterio, y_rasterio))
print("Same as manual calculation?", (x_manual, y_manual) == (x_rasterio, y_rasterio))

valid = arr != nodata
print("valid cell count:", int(valid.sum()))
print("raw minimum:", arr.min())
print("valid minimum (m):", arr[valid].min())

safe = np.where(valid, arr, -np.inf)
flat_index = safe.argmax()
max_row, max_col = np.unravel_index(flat_index, arr.shape)
print("highest valid cell (row, col):", (max_row, max_col))
print("highest valid elevation (m):", arr[max_row, max_col])

**Why use a mask?** `-9999` is missing because we declared it to be missing in **this** dataset. In another raster, zero may be a valid temperature and should be kept.

`arr[valid]` collects the included values into a **one-dimensional array** for statistics; it no longer preserves their original row and column positions. `np.where(valid, arr, -np.inf)` keeps the **two-dimensional grid shape**, so `argmax` followed by `np.unravel_index(..., arr.shape)` can recover the maximum's row and column. The temporary array excludes invalid cells from the maximum search and does **not** alter `arr`.


## 6 · Plot values in projected coordinates

`masked=True` uses the dataset's declared NoData. Plotting with the transform allows the axes to show projected easting and northing, not bare array indices.

**Colourbar check:** A valid elevation colourbar must use actual metre values (about 200–240 m), not a normalized 0–1 display scale. We use Matplotlib `imshow` with the Rasterio bounds to keep the original values and projected coordinates together.

In [ ]:
with rasterio.open(tif_path) as src:
    masked = src.read(1, masked=True)
    fig, ax = plt.subplots(figsize=(8, 6))

    # imshow keeps the true elevation values for the colorbar.
    # The extent places the image using the raster's projected bounds.
    b = src.bounds
    im = ax.imshow(
        masked, extent=[b.left, b.right, b.bottom, b.top],
        origin="upper", cmap="terrain", interpolation="nearest"
    )
    px, py = src.xy(int(max_row), int(max_col))
    ax.scatter([px], [py], marker="x", s=130, c="black", linewidths=2)
    ax.annotate("highest valid cell", (px, py), xytext=(8, 8),
                textcoords="offset points", fontsize=10, color="black")

    # im is built from the physical elevation array, not a 0-1 rendering.
    fig.colorbar(im, ax=ax, label="Synthetic elevation (m)", shrink=0.8)
    ax.set_xlabel("Easting (m) · EPSG:32633")
    ax.set_ylabel("Northing (m) · EPSG:32633")
    ax.set_title("Georeferenced synthetic elevation raster")
    fig.tight_layout()
    plt.show()

**Guided self-check:** `shape=(4, 4)`, `arr[2,1]=213 m`, the cell centre is `(600150, 5339750)` metres in EPSG:32633, the count of valid cells is `15`, the valid minimum is `200 m`, and the highest valid elevation is `240 m` at `(row=2,col=2)`. The one `-9999` cell should not appear as a very low terrain value on the plot.

## Part B — Guided real Vienna DEM · Sections 7–9

We now **repeat** the core workflow on a genuine elevation raster. This part is demonstrated step by step, just like Part A. The grid dimensions, value range, spatial extent and CRS come from the **actual GeoTIFF**, not from our invented 4 × 4 array.

The instructor supplies `W04_Vienna_DEM_Clip.tif` (a genuine elevation clip, *not* a hillshade picture). Its reported statistics describe this specific area, not Vienna as a whole.

The **second** real file, `W04_Vienna_WorldCover_Clip.tif`, is intentionally **not analysed here**: it is reserved for your Independent Checkpoint in Part C.

In [ ]:
# 7 · Find the instructor-provided authentic Vienna DEM
real_dem_path = Path('W04_Vienna_DEM_Clip.tif')

if real_dem_path.is_file():
    print('FOUND genuine Vienna DEM:', real_dem_path.resolve())
else:
    print('MISSING authentic Vienna DEM:', real_dem_path)
    print('Ask the instructor for the real cropped GeoTIFF; do not substitute synthetic data.')

### 8 · Inspect metadata in a genuine Vienna DEM

This is the **guided** real-data example. Use familiar Rasterio methods to inspect `shape`, `CRS`, `resolution`, `bounds`, `nodata` and valid pixel count. Values now refer to *actual* terrain rather than invented elevations. Do not assume its planar CRS or vertical reference equals the synthetic EPSG:32633 example.

**What to notice:** The file's map extent and pixel size come from its own metadata. The source documentation should be consulted for its vertical elevation reference.

In [ ]:
# Read the real DEM; do not substitute made-up data if the file is unavailable.
real_dem_info = None
if real_dem_path.is_file():
    with rasterio.open(real_dem_path) as src:
        real_dem = src.read(1, masked=True)
        real_dem_info = {
            'shape': src.shape,
            'count': src.count,
            'crs': str(src.crs),
            'resolution': src.res,
            'bounds': src.bounds,
            'nodata': src.nodata,
            'valid_cells': int(np.ma.count(real_dem)),
        }
    for name, result in real_dem_info.items():
        print(f'{name}: {result}')
else:
    print('Part B DEM unavailable: put an authentic instructor-prepared GeoTIFF at', real_dem_path)

### 9 · Map real Vienna terrain and calculate valid statistics

Using the **guided** DEM, make a georeferenced elevation plot with a continuous colourbar and calculate the valid minimum, maximum and mean elevation. The code also marks one valid location on the map to connect row/column with real coordinates; the main task is interpreting the spatial data, not re-deriving the cell centre.

**Read, select, summarise:**

1. **Read:** `src.read(1, masked=True)` keeps the file's values and mask; mask `True` means excluded.
2. **Select:** `~np.ma.getmaskarray(values)` marks unmasked cells as valid. `valid &= np.isfinite(np.asarray(values))` also requires their values to be finite.
3. **Summarise:** `np.asarray(values)[valid]` selects the elevations used for the minimum, maximum and mean.

**Interpretation:** These statistics describe only the actual DEM clip. Elevation is height measured relative to a chosen zero level, or **vertical reference**. The supplied Vienna DEM uses **Wiener Null**. A negative elevation lies below that reference level and can be valid; missing cells are identified by the file's mask.


In [ ]:
# Run only when the real elevation file is available.
if real_dem_path.is_file():
    with rasterio.open(real_dem_path) as src:
        values = src.read(1, masked=True)
        valid = ~np.ma.getmaskarray(values)
        # Also rule out unexpected non-finite values (if any).
        valid &= np.isfinite(np.asarray(values))
        if not np.any(valid):
            print('This real DEM clip contains no valid finite elevation values.')
        else:
            elevations = np.asarray(values)[valid]
            print('Valid elevation min:', float(elevations.min()))
            print('Valid elevation max:', float(elevations.max()))
            print('Valid elevation mean:', float(elevations.mean()))
            # Choose one real valid cell near the centre of the data footprint.
            coords = np.argwhere(valid)
            middle = np.asarray(src.shape) / 2.0
            row, col = map(int, coords[np.argmin(((coords-middle)**2).sum(axis=1))])
            x_real, y_real = src.xy(row, col)
            print(f'Example real cell: row={row}, col={col}, value={float(values[row,col]):.3f}, x={x_real}, y={y_real}, CRS={src.crs}')
            fig, ax = plt.subplots(figsize=(8, 6))
            bounds = src.bounds
            im = ax.imshow(values, origin='upper',
                extent=[bounds.left, bounds.right, bounds.bottom, bounds.top],
                cmap='terrain', interpolation='nearest')
            ax.scatter([x_real], [y_real], c='black', marker='x', s=90)
            fig.colorbar(im, ax=ax, label='Elevation (source vertical reference)', shrink=0.8)
            ax.set_xlabel(f'X coordinate ({src.crs})')
            ax.set_ylabel(f'Y coordinate ({src.crs})')
            ax.set_title('Real Vienna DEM clip — source: Stadt Wien')
            fig.tight_layout()
            plt.show()
else:
    print('Skipping real DEM map/stats: an authentic clipped GeoTIFF is required.')

### Guided Part B — interpretation check

Verify that your Vienna DEM map has projected/map coordinates, a continuous elevation colourbar and a source citation. Record the **actual** `shape`, `CRS`, resolution, bounds, NoData, valid min/max/mean and an example cell's value and location. If `W04_Vienna_DEM_Clip.tif` is missing, ask for the authentic file rather than inventing statistics.

**Part B deliverable:** one real Vienna DEM map and a small metadata/statistics summary.

The next section switches to an **independent** task with a different real GeoTIFF. It is deliberately categorical rather than quantitative, so not every summary used for the DEM will make sense.

## Part C — Independent checkpoint · Real ESA WorldCover

**Dataset:** `W04_Vienna_WorldCover_Clip.tif`, a genuine, instructor-prepared crop of **ESA WorldCover 2021 Map** (not the DEM and not simulated data).

This raster stores **land-cover class codes**, not metres of elevation. Use the product's category meanings to interpret its cells:

| Code | Land-cover class | Code | Land-cover class |
|---:|---|---:|---|
| 10 | Tree cover | 20 | Shrubland |
| 30 | Grassland | 40 | Cropland |
| 50 | Built-up | 60 | Bare / sparse vegetation |
| 70 | Snow and ice | 80 | Permanent water bodies |
| 90 | Herbaceous wetland | 95 | Mangroves |
| 100 | Moss and lichen |  |  |

Only classes **actually present** in the clipped raster belong in your results. Do not take the mean of class IDs. Class proportions by pixel count are **not automatically areas in square metres**, especially for data in a geographic CRS.

Unlike Guided Part B, **the analysis and plot are not already completed for you**. You should adapt the familiar reading, masking, summarising and mapping techniques to a categorical dataset.

### Section 10 · Run the supplied setup and toy example

This section prepares the tools for the independent analysis of the real `W04_Vienna_WorldCover_Clip.tif`. Run the supplied cells below **in order**:

1. **Setup:** check the WorldCover input path and define the class labels and colours.
2. **Toy counts:** run the synthetic 3 × 3 grid, its valid mask, and the counts and shares.
3. **Helper:** run the `plot_categorical` function definition once.
4. **Toy map:** run the provided function call and inspect its output.

Verify eight valid cells, counts `4, 2, 2`, shares `50%, 25%, 25%`, a three-class legend and a blank NoData cell.

Then continue to **§11** and write your own analysis of the real WorldCover file, using its own metadata, mask and observed classes. Use the supplied `plot_categorical` helper; **you do not need to rewrite it**. The setup and toy cells provide the method, while your §11 code produces the real results.


In [ ]:
# Independent checkpoint — inputs and reference codebook only
worldcover_path = Path('W04_Vienna_WorldCover_Clip.tif')
worldcover_labels = {
    10: 'Tree cover', 20: 'Shrubland', 30: 'Grassland',
    40: 'Cropland', 50: 'Built-up', 60: 'Bare / sparse vegetation',
    70: 'Snow and ice', 80: 'Permanent water bodies',
    90: 'Herbaceous wetland', 95: 'Mangroves', 100: 'Moss and lichen',
}
# Reference palette provided so the challenge tests analysis, not colour memorisation.
worldcover_colors = {
    10: '#006400', 20: '#ffbb22', 30: '#ffff4c',
    40: '#f096ff', 50: '#fa0000', 60: '#b4b4b4',
    70: '#f0f0f0', 80: '#0064c8', 90: '#0096a0',
    95: '#00cf75', 100: '#fae6a0',
}

worldcover_source = "ESA WorldCover project 2021 / Contains modified Copernicus Sentinel data (2021) processed by ESA WorldCover consortium"

if worldcover_path.is_file():
    print('Checkpoint file found:', worldcover_path.resolve())
else:
    print('MISSING real WorldCover file:', worldcover_path)
    print('Obtain the authentic Map GeoTIFF clip from the instructor before completing Part C.')


### Toy practice · Count classes and make a categorical map

This **synthetic 3 × 3 grid** demonstrates a reusable plotting template. Its codes
use familiar class labels and colours, but its values are invented. The centre
cell is NoData. Class `30` is in the reference codebook and is absent from the grid.

`np.unique(..., return_counts=True)` returns the observed codes and their counts
in matching order. Select valid cells first, then divide each count by the total
number of valid cells to obtain its percentage.

In [ ]:
# Synthetic example: eight valid cells and one declared NoData cell.
toy_data = np.array([
    [10, 10, 50],
    [10, -9999, 50],
    [10, 80, 80],
], dtype=np.int16)
toy_nodata = -9999
toy_valid = toy_data != toy_nodata

toy_labels = {
    10: "Tree cover", 30: "Grassland",
    50: "Built-up", 80: "Permanent water bodies",
}
toy_colors = {
    10: "#006400", 30: "#ffff4c",
    50: "#fa0000", 80: "#0064c8",
}

# Each class and its count occupy the same position in the two outputs.
classes, counts = np.unique(
    toy_data[toy_valid], return_counts=True
)
shares = 100 * counts / counts.sum()
print("Valid cells:", int(toy_valid.sum()))
print("Classes:", classes)
print("Counts:", counts)
print("Shares (%):", shares)
for class_code, count, share in zip(classes, counts, shares):
    print(f"{class_code}: {toy_labels[class_code]} — {count} cells, {share:.1f}%")


### A reusable plotting template

Run the helper below once. Its inputs are the **2D code array**, a **valid mask**
(`True` includes a cell), the **observed classes**, the label/colour dictionaries,
and the raster's own extent and CRS. `extent` is `[left, right, bottom, top]`.

- `np.ma.masked_all` starts a display with every cell excluded; the loop fills
  valid cells with colour positions `0, 1, …`. The original code array is unchanged.
- `ListedColormap` uses the exact colour list. The positions have equal-width
  colour slots, so numeric gaps between class codes do not affect their colours.
- `Patch` creates one legend key per observed class. The map and legend look up
  colours from the same dictionary, so a missing class cannot shift the mapping.

The helper returns `fig, ax`, as in the earlier Matplotlib examples. It draws the
provided data; your checkpoint code still obtains the data, mask and classes.

In [ ]:
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch


def plot_categorical(data, valid, classes, labels, colors,
                     extent, crs, title, source):
    """Plot a north-up raster using its observed valid classes.

    valid is True for included cells; classes comes from np.unique(data[valid]).
    extent is [left, right, bottom, top] in the supplied CRS.
    """
    # Map class codes to colour positions 0, 1, ...; keep other cells masked.
    display = np.ma.masked_all(data.shape, dtype=int)
    for colour_position, class_code in enumerate(classes):
        display[valid & (data == class_code)] = colour_position

    # Look up colours by code, so an absent class cannot shift other colours.
    palette = [colors[class_code] for class_code in classes]
    cmap = ListedColormap(palette)
    fig, ax = plt.subplots(figsize=(8.4, 6.4))
    ax.imshow(display, cmap=cmap,
              vmin=-0.5, vmax=len(classes) - 0.5,
              extent=extent, origin="upper", interpolation="nearest")

    handles = [Patch(facecolor=colors[class_code], label=labels[class_code])
               for class_code in classes]
    ax.legend(handles=handles, title="Observed classes", frameon=False,
              loc="upper center", bbox_to_anchor=(0.5, -0.22),
              ncol=3, fontsize=14, title_fontsize=14)
    ax.set_title(title, fontsize=16, pad=12)
    ax.set_xlabel(f"X coordinate ({crs})", fontsize=14)
    ax.set_ylabel(f"Y coordinate ({crs})", fontsize=14)
    ax.set_xticks([extent[0], (extent[0] + extent[1]) / 2, extent[1]])
    ax.set_yticks([extent[2], (extent[2] + extent[3]) / 2, extent[3]])
    ax.tick_params(labelsize=14)
    ax.ticklabel_format(style="plain", useOffset=False)
    fig.subplots_adjust(left=0.15, right=0.98, bottom=0.30, top=0.90)
    fig.text(0.5, 0.015, source, ha="center", va="bottom", fontsize=14)
    return fig, ax


In [ ]:
# The toy has 100 m cells and a known, north-up spatial reference.
toy_extent = [600000, 600300, 5339700, 5340000]
toy_crs = "EPSG:32633"
toy_title = "Categorical raster: 8 valid cells"
toy_source = "Synthetic example"

toy_fig, toy_ax = plot_categorical(
    toy_data, toy_valid, classes,
    toy_labels, toy_colors, toy_extent,
    toy_crs, toy_title, toy_source
)

# Code labels and cell boundaries help us check this small toy grid.
for row, col in np.argwhere(toy_valid):
    x = 600000 + (col + 0.5) * 100
    y = 5340000 - (row + 0.5) * 100
    toy_ax.text(x, y, str(toy_data[row, col]), ha="center", va="center",
                color="white", fontsize=22, fontweight="bold")
for x in [600100, 600200]:
    toy_ax.axvline(x, color="white", linewidth=1.5)
for y in [5339800, 5339900]:
    toy_ax.axhline(y, color="white", linewidth=1.5)

toy_fig.savefig(workdir / "toy_categorical_map.png", dpi=180)
plt.show()


**Check the toy output:** eight valid cells; codes `10, 50, 80`; counts `4, 2, 2`;
shares `50%, 25%, 25%`. The centre remains blank. Grassland (`30`) does not appear
in the legend because it is absent from the valid cells.

For the independent task, obtain the actual file's values, valid mask, observed
classes, extent and CRS before using `plot_categorical`. Use its class labels and
colours, and supply the WorldCover source credit. Your counts, percentages and
interpretation must come from your own analysis of the real file.


### Section 11 · Independent analysis of the real WorldCover raster

Complete the analysis in the stages below. **Use the code cell directly under each stage and run it before moving on.** Keeping the steps separate will make it easier to inspect your intermediate results and find mistakes.

You can reuse `worldcover_path`, `worldcover_labels`, `worldcover_colors`, `worldcover_source`, `workdir` and `plot_categorical` from §10.


#### 11.1 · Inspect the raster metadata

**Goal:** understand the real file before analysing its values.

In the next code cell:
1. Open `worldcover_path` with Rasterio.
2. Inspect and print the raster **shape, band count, CRS, resolution, bounds and NoData value**.
3. Save the **CRS** and **bounds** in variables because you will reuse them when you make the map.

This is the same kind of metadata check you used for the Vienna DEM in §8.


In [ ]:
# 11.1 · Inspect the WorldCover metadata
# Suggested variables to keep for later: worldcover_crs, worldcover_bounds

# Write your code here.


#### 11.2 · Read band 1 and identify valid cells

**Goal:** separate the land-cover observations you can analyse from cells that should be excluded.

In the next code cell:
1. Read **band 1** with masking turned on and store it as `worldcover`.
2. Create a Boolean mask called `worldcover_valid`, where `True` means the cell is included.
3. Count and print the number of valid cells.
4. Check the array shape so that you know the mask still matches the raster grid.

Use the same masking idea as in the guided DEM section. Do not replace the file's mask with an assumed value.


In [ ]:
# 11.2 · Read band 1 and build the valid-cell mask
# Suggested variables: worldcover, worldcover_valid

# Write your code here.


#### 11.3 · Find the observed classes, counts and percentage shares

**Goal:** summarise a categorical raster using class frequencies rather than a mean.

In the next code cell:
1. Use **only valid cells**.
2. Find the distinct class codes that actually occur in the clip and count them.
3. Calculate each class's percentage of all valid cells.
4. Print a small results table with **code, class label, count and percentage**.
5. Check that the percentages add up to approximately **100%**.

`np.unique(..., return_counts=True)` from the toy example is useful here. Suggested variable names are `wc_classes`, `wc_counts` and `wc_shares`.


In [ ]:
# 11.3 · Count observed classes and calculate shares
# Suggested variables: wc_classes, wc_counts, wc_shares

# Write your code here.


#### 11.4 · Identify the most common class

**Goal:** turn the class-frequency results into one clear finding.

In the next code cell:
1. Find which observed class has the largest count.
2. Print its **code, official label and percentage share**.
3. Keep this result for your written interpretation below.

Use the counts and shares you created in 11.3 rather than reading the raster again.


In [ ]:
# 11.4 · Find the most common land-cover class

# Write your code here.


#### 11.5 · Make a georeferenced categorical map

**Goal:** visualise the real class pattern in the raster's own coordinate system.

In the next code cell:
1. Convert the Rasterio bounds to an `extent` in the order **[left, right, bottom, top]**.
2. Call the supplied `plot_categorical(...)` helper using the real data, valid mask and observed classes.
3. Use `worldcover_labels`, `worldcover_colors` and `worldcover_source` from §10.
4. Give the map a clear title, save it in `week4_outputs/`, and display it.
5. Check that the legend contains only classes actually present in this clip and that NoData remains blank.

You do **not** need to rewrite the plotting helper.


In [ ]:
# 11.5 · Plot the real WorldCover raster
# Suggested variables: wc_extent, wc_fig, wc_ax

# Write your code here.


### 11.6 · Your results and interpretation

After running the five code cells above, write **3–5 sentences** that interpret your real WorldCover result.

Include:
- the raster's CRS, resolution, dimensions and valid-cell count;
- the **most common land-cover class** and its percentage;
- why taking the **mean of the numeric class codes** would not be meaningful;
- why a percentage of pixels should not automatically be reported as an area in square metres without considering the raster's spatial reference and cell geometry.

**Map check:** categorical colours, matching legend, geographic/map coordinates and WorldCover source credit.

**Before submission:** run the notebook from top to bottom, make sure your §11 cells show their outputs, and save the `.ipynb` before following the Moodle submission instructions.
